In [ ]:
# -------------------- INSTALL PACKAGES --------------------
!pip install praw transformers torch pandas requests beautifulsoup4 --quiet

# -------------------- Ignoring  WARNINGS --------------------
from bs4 import MarkupResemblesLocatorWarning
import warnings
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

# -------------------- IMPORTS --------------------
import praw
import requests
import pandas as pd
import math
import torch
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# -------------------- REDDIT CRED --------------------
reddit = praw.Reddit(
    client_id='flIZiYMjDnPJh8-LvWWF8g',
    client_secret='pKMbkiYpMbErMt7-lQqLxNXVSW2Wzg',
    user_agent='meme-coin-reddit-scraper'
)

# -------------------- SENTIMENT MODEL --------------------
model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
labels = ["Negative", "Neutral", "Positive"]
score_map = {"Positive": 1, "Neutral": 0, "Negative": -1}

# -------------------- COINS OF INTREST --------------------
coin_queries = {
    'DOGE': ['DOGE', 'Dogecoin', '$DOGE'],
    'BTC': ['BTC', 'Bitcoin', '$BTC'],
    'PEPE': ['PEPE', 'Pepe Coin', '$PEPE'],
    'CHILLGUY': ['CHILLGUY', 'Just a chill guy', '$CHILLGUY'],
    'BONK': ['BONK', 'Bonk Coin', '$BONK']
}
#--------------------- SUBREDDITS -------------------------
subreddits = [
    'crypto_moon_shots','satoshi_street_bets','dogecoin','shiba_army','pepe_coin',
    'bonk_coin','wall_street_bets_crypto','alt_street_bets','eth_trader','cc_meta',
    'penny_stocks_crypto','shitcoin_bets','solana_meme_coins','memetic_traders','token_scout',
    'crypto_horde','all_crypto_bets','crypto_general','coin_market_doors','wsb_dapp',
    'cryptocurrency','crypto_markets','crypto_currency_trading','safe_moon',
    'cryptomemes','cryptocurrency_memes','cryptojokes','cryptonews','cryptotrading',
    'cryptoinvesting','cryptotalk','cryptopumping','cryptopump','cryptosignals'
]
#------------------ REMOVING BIASING ---------------------
biased_subreddits = ['PEPECoin','bonk_coin','safemoon','dogecoin']
days_back = 15
since_date = datetime.utcnow() - timedelta(days=days_back)
max_posts_per_coin = 1000

# -------------------- TOKENIZING--------------------
def classify_sentiment_bert(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    outputs = model(**inputs)
    scores = outputs.logits.detach().numpy()[0]
    probs = torch.nn.functional.softmax(torch.tensor(scores), dim=0).numpy()
    return labels[probs.argmax()]
#-------------- GIVING SCORES TO THE POSTS--------------
def compute_weighted_score(row):
    base = score_map.get(row["Sentiment"], 0)
    up = max(0, row.get("Upvotes", 0))
    w = math.log1p(up)
    if row["Subreddit"].lower() in biased_subreddits:
        w *= 0.5
    return base * w
#--------------- 4CHAN SCRAPPING FUNCTION-------------
def scrape_4chan(coin_queries, days_back, limit_per_coin=300):
    after = datetime.utcnow() - timedelta(days=days_back)
    posts = []
    try:
        catalog = requests.get("https://a.4cdn.org/biz/catalog.json").json()
    except Exception as e:
        print("Error fetching 4chan catalog:", e)
        return pd.DataFrame(columns=[
            "Date", "Subreddit", "Title", "Text", "Upvotes", "Post URL", "Coin"
        ])
    for coin, queries in coin_queries.items():
        count = 0
        for page in catalog:
            for th in page["threads"]:
                if count >= limit_per_coin:
                    break
                tid = th["no"]
                try:
                    thread = requests.get(f"https://a.4cdn.org/biz/thread/{tid}.json").json()
                    for post in thread.get("posts", []):
                        t = datetime.utcfromtimestamp(post["time"])
                        if t < after:
                            continue
                        txt = BeautifulSoup(post.get("com", ""), "html.parser").get_text()
                        subj = post.get("sub", "")
                        full = (subj + " " + txt).lower()
                        if any(q.lower() in full for q in queries):
                            posts.append([t, "/biz/", subj, txt, post.get("replies", 0),
                                          f"https://boards.4channel.org/biz/thread/{tid}", coin])
                            count += 1
                            if count >= limit_per_coin:
                                break
                except:
                    continue
            if count >= limit_per_coin:
                break
    return pd.DataFrame(posts, columns=[
        "Date", "Subreddit", "Title", "Text", "Upvotes", "Post URL", "Coin"
    ])

# -------------------- SCRAPING REDDIT --------------------
all_posts = []
for coin, queries in coin_queries.items():
    count = 0
    for sub in subreddits:
        try:
            sr = reddit.subreddit(sub)
            for q in queries:
                for post in sr.search(q, sort='new', time_filter='month', limit=300):
                    if count >= max_posts_per_coin:
                        break
                    t = datetime.utcfromtimestamp(post.created_utc)
                    if t < since_date:
                        continue
                    all_posts.append([t, sub, post.title, post.selftext, post.score, post.url, coin])
                    count += 1
                if count >= max_posts_per_coin:
                    break
            if count >= max_posts_per_coin:
                break
        except Exception as e:
            print(f"Skipping subreddit '{sub}' due to error: {e}")
            continue

df_red = pd.DataFrame(all_posts, columns=[
    "Date", "Subreddit", "Title", "Text", "Upvotes", "Post URL", "Coin"
])

# -------------------- SCRAPE 4CHAN --------------------
df_4c = scrape_4chan(coin_queries, days_back, limit_per_coin=300)

# -------------------- COMBINE & SORT --------------------
df = pd.concat([df_red, df_4c], ignore_index=True)
df.sort_values(["Coin", "Date"], inplace=True)
df["Post"] = df["Title"].fillna("") + ". " + df["Text"].fillna("")

# -------------------- SENTIMENT ANALYSIS --------------------
print("🧠 Performing sentiment analysis...")
df["Sentiment"] = df["Post"].apply(classify_sentiment_bert)
df["Weighted_Score"] = df.apply(compute_weighted_score, axis=1)

# -------------------- SAVE DETAILED CSV --------------------
df.to_csv("combined_meme_posts_with_sentiment.csv", index=False)

# -------------------- SUMMARY (NORMALIZED) --------------------
summary = df.groupby("Coin").agg(
    Total_Weighted_Score=("Weighted_Score", "sum"),
    Num_Posts=("Weighted_Score", "count")
)
summary["Average_Weighted_Score"] = summary["Total_Weighted_Score"] / summary["Num_Posts"]
summary.sort_values("Average_Weighted_Score", ascending=False, inplace=True)
summary.to_csv("sentiment_summary_normalized.csv", index=True)

# -------------------- PRICE PREDICTION LOGIC (CAN BE ADJUSTED AFTER BACKTESTING) --------------------
def predict_price_direction(score):
    if score > 1.0:
        return "🚀 Likely to go UP"
    elif score < 0.2:
        return "📉 May go DOWN"
    else:
        return "⚖️ Possibly STABLE"

summary["Prediction"] = summary["Average_Weighted_Score"].apply(predict_price_direction)

# -------------------- SAVE FINAL PREDICTIONS --------------------
summary.to_csv("final_prediction_summary.csv", index=True)

# -------------------- DISPLAY --------------------
print("\n📈 Final Price Prediction Based on Sentiment:")
print(summary[["Average_Weighted_Score", "Prediction"]])


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Skipping subreddit 'crypto_moon_shots' due to error: Redirect to /subreddits/search
Skipping subreddit 'satoshi_street_bets' due to error: Redirect to /subreddits/search


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/l

Skipping subreddit 'shiba_army' due to error: Redirect to /subreddits/search


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/l

...

Skipping subreddit 'cryptocurrency_memes' due to error: received 404 HTTP response
Skipping subreddit 'cryptojokes' due to error: Redirect to /subreddits/search


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/l

Skipping subreddit 'cryptotalk' due to error: received 404 HTTP response
Skipping subreddit 'cryptopumping' due to error: received 404 HTTP response


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/l

🧠 Performing sentiment analysis...

📈 Final Price Prediction Based on Sentiment:
          Average_Weighted_Score          Prediction
Coin                                                
BONK                    2.242357   🚀 Likely to go UP
CHILLGUY                0.553968  ⚖️ Possibly STABLE
DOGE                    0.544020  ⚖️ Possibly STABLE
PEPE                    0.491966  ⚖️ Possibly STABLE
BTC                     0.191250       📉 May go DOWN
